In [20]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.sparse import hstack

from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor


In [3]:
sns.set_theme(style="white")

In [41]:

# Load the Train Parquet file for EDA
raw_train_df = pd.read_parquet("../datasets/raw_train_oct_2024.snappy.parquet")

print("Dataset loaded successfully!")
raw_train_df.iloc[:, :15].head(20)  # View first few rows


Dataset loaded successfully!


,stock_item_id,last_date_seen,first_date_seen,days_to_sell,first_retailer_asking_price,last_retailer_asking_price,can_home_deliver,reviews_per_100_advertised_stock_last_12_months,segment,seats,doors,co2_emission_gpkm,top_speed_mph,zero_to_sixty_mph_seconds,engine_power_bhp
0,b6598141a3d1dca4825a72e0f55ddfeb0fb55cabbc14da...,2024-10-12,2024-10-09,3,3695,3695,False,1.0,Independent,5.0,5.0,124.0,107.0,12.7,84.0
1,acdde4604ab932227b68abff979123a3f0b5d26d8b711e...,2024-10-11,2024-09-13,28,18998,18298,False,6.3,Franchise,5.0,5.0,130.0,119.0,None,129.0
2,0d29437822e059c038fc7f6385d9c348ba3727509add9a...,2024-10-05,2024-08-23,43,13990,13990,False,0.2,Independent,5.0,5.0,115.0,124.0,10.0,124.0
3,bbd82c7e47cba47f96ac59c7022ff1ca485c869f660778...,2024-10-15,2024-09-30,15,8174,7990,False,0.3,Independent,4.0,3.0,125.0,126.0,None,123.0
4,2515373efa5d36e0220cd42cc1cea23686e547cad17ca3...,2024-10-07,2024-08-15,53,9450,8750,False,7.3,Independent,5.0,5.0,146.0,120.0,None,123.0
5,cc8b1398b26941edd01b3280dda7a8fa4228a44fca127d...,2024-10-02,2024-09-04,28,7897,7897,False,4.4,Independent,5.0,5.0,158.0,109.0,11.8,110.0
6,10038453f1d28cf2d54764415b16831dbd4729005143c8...,2024-10-26,2024-10-01,25,23500,23000,False,0.7,Franchise,5.0,5.0,147.0,130.0,8.7,148.0
7,768836998ab25ca0284a4ac9aaf0b30c01dd6a1cdb1154...,2024-10-08,2024-09-26,12,10495,10495,False,45.0,Independent,4.0,2.0,124.0,140.0,None,187.0
8,14df89a70ac4058f6f65234a96ff50cfdfc0fcb42ffb4e...,2024-10-02,2024-09-27,5,5250,5250,False,4.6,Independent,5.0,5.0,193.0,110.0,None,148.0
9,ebbe035af17cd4ab562eb71e81da90633cc9b6ee6da588...,2024-10-29,2024-09-23,36,39000,39000,False,0.7,Independent,5.0,5.0,177.0,155.0,None,261.0


In [24]:
# Sample 30% of the data (replace 0.3 with your desired fraction)
sampled_df = raw_train_df.sample(frac=0.01, random_state=42)

X = sampled_df.drop(columns=['days_to_sell'])  # Drop target from features
y = sampled_df['days_to_sell']  # Target variable

## Transformation pipeline utils

In [17]:
# Replace None with Nan Transformer.
def replace_none_with_nan(df: pd.DataFrame) -> pd.DataFrame:
    """Replaces None with np.nan in categorical columns."""
    df = df.copy()
    categorical_cols = df.select_dtypes(include=['object']).columns
    df[categorical_cols] = df[categorical_cols].apply(lambda col: col.map(lambda x: np.nan if x is None else x))
    return df


def drop_cat_features(df: pd.DataFrame) -> pd.DataFrame:
    """Drop features not required"""
    df = df.copy()
    columns_to_drop = ["stock_item_id", "last_date_seen", "first_date_seen", "derivative_id", "first_registration_date"]
    return df.drop(columns=columns_to_drop, errors='ignore')


def convert_columns_to_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """Converts specified object columns to float/numeric type without dropping them."""
    df = df.copy()
    
    convert_columns = ["zero_to_sixty_mph_seconds", "engine_power_bhp", "fuel_economy_wltp_combined_mpg",
                       "battery_usable_capacity_kwh", "length_mm", "insurance_group", "plate"]
    
    # Ensure we only select columns that exist in the DataFrame
    existing_columns = list(set(df.columns) & set(convert_columns))
    
    if existing_columns:  # Only apply conversion if columns exist
        df[existing_columns] = df[existing_columns].apply(pd.to_numeric, errors='coerce')

    return df



In [18]:


# Global definition of the text features (default feature list)

# Initialize TfidfVectorizer for merging features
tfidf_vectorizer = TfidfVectorizer()

def merge_make_model(df: pd.DataFrame) -> np.ndarray:
    """Merges car features into a single string column and applies TF-IDF Vectorizer."""
    df = df.copy()  # Ensure original DataFrame is not modified
    
    # Merge the relevant columns into a single string
    merged_column = df['make'] + " " + df['model'] + " " + df['generation'] + " " + df['derivative']
    
    # Apply the TF-IDF vectorizer to the merged column
    return tfidf_vectorizer.fit_transform(merged_column.fillna(''))  # Handle missing values by replacing them with an empty string


In [46]:
DEFAULT_ZERO_NUM_FEATURES = ['battery_range_miles', 'battery_usable_capacity_kwh']
DEFAULT_NON_ZERO_NUM_FEATURES = ['first_retailer_asking_price',
 'last_retailer_asking_price',
 'reviews_per_100_advertised_stock_last_12_months',
 'seats',
 'doors',
 'co2_emission_gpkm',
 'top_speed_mph',
 'zero_to_sixty_mph_seconds',
 'engine_power_bhp',
 'fuel_economy_wltp_combined_mpg',
 'length_mm',
 'boot_space_seats_up_litres',
 'insurance_group',
 'plate',
 'odometer_reading_miles',
 'adjusted_retail_amount_gbp',
 'predicted_mileage',
 'number_of_images',
 'advert_quality']
DEFAULT_HOTENCODE_FEATURES = ['can_home_deliver', 'transmission_type', 'manufacturer_approved', 'segment', 'body_type', 'fuel_type', 'colour', 'first_image_label', 'drivetrain' ]
DEFAULT_CAR_FEATURES = ['make', 'model', 'generation', 'derivative']

# Data preprocessing pipeline
data_preparation_pipeline = Pipeline([
    ('drop_columns', FunctionTransformer(drop_cat_features)), # Drop id and date columns
    ('replace_none', FunctionTransformer(replace_none_with_nan, validate=False)),  # Custom transformer to handle 'None'
    ('convert_2numeric', FunctionTransformer(convert_columns_to_numeric))
])


# Define a numerical transformer for the column transformer
numerical_transformer = ColumnTransformer(
    transformers=[
        ('zero_impute', SimpleImputer(strategy='constant', fill_value=0), DEFAULT_ZERO_NUM_FEATURES), # Zero imputation for specific columns
        ('imputation', SimpleImputer(strategy='mean'), DEFAULT_NON_ZERO_NUM_FEATURES) # Mean imputation for other numerical columns
    ]
)

# Numerical pipeline without outlier removal
numeric_pipeline = Pipeline([
    ('imputer', numerical_transformer),  # Handle missing values
    ('scaler', PowerTransformer())  # Scale numeric features
])

# Define ColumnTransformer without set_output inside the tuples
categorical_transformer = ColumnTransformer(
    [
        ('he_all', OneHotEncoder(sparse_output=False, min_frequency=0.05, handle_unknown="infrequent_if_exist"), DEFAULT_HOTENCODE_FEATURES),
        ('he_postcode', OneHotEncoder(sparse_output=False, min_frequency=0.01, handle_unknown="infrequent_if_exist"), ["postcode_area"]),
        ('oe_price_indicator', OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ["price_indicator_rating"]),
        ('attention_text', CountVectorizer(), 'attention_grabber'),
        ('car_description', FunctionTransformer(merge_make_model, validate=False), DEFAULT_CAR_FEATURES)
    ]
)

# Categorical pipeline
categorical_pipeline = Pipeline([
    ('emty_string_imputer', SimpleImputer(strategy='constant', fill_value="").set_output(transform='pandas')),  # Handle missing values for categorical
    ('categorical_encoder', categorical_transformer),
])


columns_preprocessor = ColumnTransformer([
    ('numeric_features', numeric_pipeline, make_column_selector(dtype_include=np.number)),
    ('categorical_features', categorical_pipeline, make_column_selector(dtype_include=["object", "bool"]))
])

# Final pipeline combining the preprocessor and classifier
training_pipeline = Pipeline([
    ('data_preparation', data_preparation_pipeline),
    ('data_preprocessor', columns_preprocessor),
    ('model_regressor', RandomForestRegressor())
])

training_pipeline

Pipeline(steps=[('data_preparation',
                 Pipeline(steps=[('drop_columns',
                                  FunctionTransformer(func=<function drop_cat_features at 0x7d99da352020>)),
                                 ('replace_none',
                                  FunctionTransformer(func=<function replace_none_with_nan at 0x7d99da351800>)),
                                 ('convert_2numeric',
                                  FunctionTransformer(func=<function convert_columns_to_numeric at 0x7d99da3520c0>))])),
                ('data_preprocess...
                                                                                                    ['price_indicator_rating']),
                                                                                                   ('attention_text',
                                                                                                    CountVectorizer(),
                                                                                                    'attention_grabber'),
                                                                                                   ('car_description',
                                                                                                    FunctionTransformer(func=<function merge_make_model at 0x7d99da3505e0>),
                                                                                                    ['make',
                                                                                                     'model',
                                                                                                     'generation',
                                                                                                     'derivative'])]))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7d99d816f090>)])),
                ('model_regressor', RandomForestRegressor())])

In [43]:
X.shape, y.shape

((1121, 42), (1121,))

In [44]:
training_pipeline.fit(X, y)

Pipeline(steps=[('data_preparation',
                 Pipeline(steps=[('drop_columns',
                                  FunctionTransformer(func=<function drop_cat_features at 0x7d99da352020>)),
                                 ('replace_none',
                                  FunctionTransformer(func=<function replace_none_with_nan at 0x7d99da351800>)),
                                 ('convert_2numeric',
                                  FunctionTransformer(func=<function convert_columns_to_numeric at 0x7d99da3520c0>))])),
                ('data_preprocess...
                                                                                                     'fuel_economy_wltp_combined_mpg',
                                                                                                     'length_mm',
                                                                                                     'boot_space_seats_up_litres',
                                                                                                     'insurance_group',
                                                                                                     'plate',
                                                                                                     'odometer_reading_miles',
                                                                                                     'adjusted_retail_amount_gbp',
                                                                                                     'predicted_mileage',
                                                                                                     'number_of_images',
                                                                                                     'advert_quality'])])),
                                                                  ('scaler',
                                                                   PowerTransformer())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7d99d816e790>)])),
                ('model_regressor', RandomForestRegressor())])

In [45]:
training_pipeline.score(X, y)

0.8562115456439108